# QLoRA Adapter Training

Train per-artist LoRA/DoRA adapters on Gemma 4 E4B.

In [1]:
from huggingface_hub import login
login()

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/gemma-4-E4B",
    local_dir="./models/gemma-4-E4B",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/home/aliozkaya/uni/467/term_project/src/models/gemma-4-E4B'

In [3]:
from peft import prepare_model_for_kbit_training

from generation.model import load_base_model
from generation.data import load_train_df
from generation.train import train_adapter
from generation.style_loss import build_style_weights, top_tokens

model, tokenizer = load_base_model()
model = prepare_model_for_kbit_training(model)

train_df = load_train_df()

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## Main adapters (LoRA)

In [4]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=False)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=False)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_lora_r8 (overwrite=True to force)


'artifacts/adapters/tool_lora_r8'

## Ablation: DoRA (same artist)

In [5]:
train_adapter(model, tokenizer, train_df, "Gojira", r=8, use_dora=True)
train_adapter(model, tokenizer, train_df, "Tool", r=8, use_dora=True)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_dora_r8 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_dora_r8 (overwrite=True to force)


'artifacts/adapters/tool_dora_r8'

## Ablation: Rank

In [6]:
train_adapter(model, tokenizer, train_df, "Gojira", r=4, use_dora=False)
train_adapter(model, tokenizer, train_df, "Gojira", r=16, use_dora=False)

[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r4 (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r16 (overwrite=True to force)


'artifacts/adapters/gojira_lora_r16'

## Remaining artists (LoRA + DoRA r=8)

Completes the lineup so the attribution table and perplexity diagonal cover all 5 artists, and the LoRA-vs-DoRA ablation generalizes beyond Gojira/Tool.

In [ ]:
for artist in ["Death", "Mastodon", "Opeth"]:
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=False, overwrite=False)
    train_adapter(model, tokenizer, train_df, artist, r=8, use_dora=True, overwrite=False)

## Experiment: Style-Weighted Loss

Up-weight artist-distinctive tokens (token-level TF-IDF) in the cross-entropy so
the adapter learns artist vocabulary, not just genre. Building/inspecting the
weights is CPU-only; only the `train_adapter` call below needs the GPU. Trains to
`*_sw` adapters so they can be A/B'd against the plain ones on attribution.

In [ ]:
SW_ARTISTS = ["Gojira", "Tool", "Death", "Mastodon", "Opeth"]
style_weights = {a: build_style_weights(a, train_df, tokenizer, text_col="clean") for a in SW_ARTISTS}

for artist in SW_ARTISTS:
    top_tokens(style_weights[artist], tokenizer, n=30)

In [9]:
for a in SW_ARTISTS:
    train_adapter(model, tokenizer, train_df, a, r=8, use_dora=False, style_weights=style_weights[a])

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[skip] adapter exists, not retraining: artifacts/adapters/gojira_lora_r8_sw (overwrite=True to force)
[skip] adapter exists, not retraining: artifacts/adapters/tool_lora_r8_sw (overwrite=True to force)


Adding EOS to train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


...

Saved: artifacts/adapters/death_lora_r8_sw (104 songs, lora, r=8_sw)


Adding EOS to train dataset:   0%|          | 0/106 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/106 [00:00<?, ? examples/s]

Step,Training Loss
5,4.391959
10,3.652728
15,3.598779
20,3.810132
25,3.328545
30,3.152900
35,2.851332
40,2.812139
45,2.781490
50,2.519411


Saved: artifacts/adapters/opeth_lora_r8_sw (106 songs, lora, r=8_sw)
